## 2. EDA with PySpark
Now, you have a table of Airbnb hosts data in your created catalog and schema. Let's understand the data a little bit before further processing the data.

Create a new notebook for EDA with PySpark:

  a) Show a histogram of numbers of reviews for each accomodation.

  b) Clean the price column. Is it numeric?

  c) Calculate the average price of accomodations per neighborhood group. In which neighborhood group is the average price highest?

  d) Show the top 10 accomodations with most reviews.

  e) Show the top 10 accomodations with best reviews.

  f) Other EDAs of your choice

In [0]:
df = spark.read.table("airbnb.hosts.bronze_airbnb")
df.limit(5).display()

In [0]:
df.select("NAME", "number_of_reviews").sort("number_of_reviews", asscending=False).display()

In [0]:
df.createOrReplaceTempView("df")

num_reviews = spark.sql("""
    SELECT NAME, 
    number_of_reviews
    FROM df
    WHERE number_of_reviews IS NOT NULL
    ORDER BY number_of_reviews DESC
     """)
num_reviews.display()

In [0]:
# a) 
import matplotlib.pyplot as plt

df_pd = df.select("number_of_reviews").toPandas()

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df_pd["number_of_reviews"].dropna(), bins=50)
ax.set_xlabel("Number of reviews")
ax.set_ylabel("Number of accommodations")
ax.set_title("Distribution of number of reviews")
plt.show()


In [0]:
df.select("price").show(10)

In [0]:
df.select("price").distinct().display()

In [0]:
# b) with sql

df.createOrReplaceTempView("df")
df_sql_price = spark.sql("""
    SELECT *,
    TRY_CAST(regexp_replace(price, '[$,]', '') AS FLOAT) as price_clean
    FROM df
""")
df_sql_price.select("price", "price_clean").show(10)

In [0]:
# b)
from pyspark.sql.functions import regexp_replace, col, when

df_pyspark = df.withColumn(
    "price_clean",
    when(
        regexp_replace(col("price"), "[^0-9.]", "") != "",
        regexp_replace(col("price"), "[^0-9.]", "").cast("float")
    ).otherwise(None)
)

df_pyspark.select("price", "price_clean").display()

In [0]:
# c) with sql
df_sql_price.createOrReplaceTempView("df_sql_price")
df_avg_sql = spark.sql("""
    SELECT neighbourhood_group, 
    ROUND(AVG(price_clean), 2) as avg_price
    FROM df_sql_price
    GROUP BY neighbourhood_group
    ORDER BY avg_price DESC
""")
df_avg_sql.display()

In [0]:
# c) with pyspark

from pyspark.sql.functions import avg, round

df_pyspark.groupBy("neighbourhood_group") \
    .agg(round(avg("price_clean"), 2).alias("avg_price")) \
    .sort("avg_price", ascending=False) \
    .display()

In [0]:
df_sql.limit(10).display()

In [0]:
# d) with sql
df.createOrReplaceTempView("df")
most_reviews_sql = spark.sql("""
    SELECT name,
    number_of_reviews
    FROM df
    WHERE number_of_reviews IS NOT NULL
    ORDER BY number_of_reviews DESC
""")
most_reviews_sql.display()


In [0]:
# d) with pyspark

df_pyspark.select("name", "number_of_reviews") \
    .filter(col("number_of_reviews").isNotNull()) \
    .sort("number_of_reviews", ascending=False) \
    .limit(10) \
    .display()

In [0]:
# e) with sql
df.createOrReplaceTempView("df")
best_reviews_sql = spark.sql("""
    SELECT name,
    review_rate_number
    FROM df
    WHERE review_rate_number IS NOT NULL
    ORDER BY review_rate_number DESC
    LIMIT 10
""")
best_reviews_sql.display()